In [2]:
import string
import jieba
import pandas as pd
import re
from ast import literal_eval
import jieba.posseg as psg
import jieba.posseg as pseg   # 分词处理
from pyltp import SentenceSplitter
pd.set_option('display.max_columns', 1000)
pd.set_option('display.width', 1000)#设置表格显示宽度
#显示所有列
pd.set_option('display.max_columns', None)
#显示所有行
pd.set_option('display.max_rows', None)



In [3]:

# 定义读取数据函数
def ReadData(DataPath, OutPutPath):
    #定义列名使其与清洗后的数据相符
    columns = ["用户ID", "用户名", "用户所属平台", "评论时间", "评论对象", "用户发表的关于文心一言或ChatGPT的评论性文本"]
    # 打开原始数据文件，使用utf-8编码方式读取
    OriginalData_text = open(DataPath, "r", encoding="utf-8")
    # 定义一个空列表，用于存储读取的数据
    OriginalData_list = []
    i = 1
    # 循环读取数据
    for line in OriginalData_text.readlines():
        #跳过第一行表头
        if i != 1:
            line = line.strip()
            linelist = line.split(",")
            OriginalData_list.append(linelist)
        i += 1
    print(len(OriginalData_list))
    # 将OriginalData_list转换为DataFrame格式，指定列名为columns
    OriginalData_DF = pd.DataFrame(data=OriginalData_list, columns=columns)
    print(OriginalData_DF.shape)
    # 将DataFrame写入到指定的输出路径
    OriginalData_DF.to_csv(OutPutPath, index=False, encoding="utf-8")


DataPath = "../all_data.csv"
OutPutPath = "../merged_file.csv"

# 调用函数
ReadData(DataPath, OutPutPath)

#保存初步处理的文件
OriginalData_DF = pd.read_csv("../merged_file.csv", encoding="utf-8")
print(OriginalData_DF.shape)

36764
(36764, 6)
(36764, 6)


In [4]:

# 进行文本清洗：清洗、去重、去除特殊字符、英文、标点符号等
def TextClean (sentence):

    # 去除Html标签符号
    pattern1 = re.compile(r'<[^>]+>', re.S)
    sentence = pattern1.sub('', str(sentence))

    # 去除 #*#
    pattern2 = re.compile(r'#[^#]+#', re.S)
    sentence = pattern2.sub('', str(sentence))

    # 去除 【*】
    pattern2 = re.compile(r'【[^】]+】', re.S)
    sentence = pattern2.sub('', str(sentence))

    # 去除 「*」
    pattern2 = re.compile(r'「[^」]+」', re.S)
    sentence = pattern2.sub('', str(sentence))

    # 去除英文、数字、标点符号、特殊字符等
    pattern3 = re.compile(r'[ 0-9a-zA-Z，—’“；：？！（）…·【】{}/、+*|《》@#￥$%&_~]')
    sentence = pattern3.sub('', sentence)
    sentence = sentence.translate(str.maketrans('', '', string.punctuation))
    sentence = sentence.strip()
    return sentence

OriginalData_DF["干净文本"] =OriginalData_DF["用户发表的关于文心一言或ChatGPT的评论性文本"].apply(TextClean)
# 删除OriginalData_DF中存在空缺值的行
OriginalData_DF.dropna(axis=0,how="any",inplace=True)

In [5]:
# 分词处理，包括去除程度词，用于制作词云图，保留名词、动名词、专有名词、时间、成语词性的词
stopwordsFile = open('../static/stopwords.txt','r+', encoding="utf-8")
stopwordsList = [word.replace("\n", "") for word in stopwordsFile.readlines()]
specialstopwords = ["文心一言", "百度","","ChatPPT","chatPPT","ChatGPT","chatGPT", "chatgpt", "谷歌","模型"]
stopwordsList =stopwordsList + specialstopwords

# 定义去除停用词函数
def cut_word_mark(sentence):
    # 指定需要保存的词性的列表
    keep_flag = ["n", "nz", "vn", 't', 'i'] 
    cuttedwords = pseg.cut(sentence)
    resultList=[]
    # 去除停用词并对词性进行标注
    for item in cuttedwords:
        if (item.word not in stopwordsList) and (item.flag in keep_flag):
            resultList.append(item.word)
    return resultList

# 生成“文本分词”列
OriginalData_DF["文本分词"] = OriginalData_DF["干净文本"].apply(cut_word_mark)


Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\UNIVER~1\AppData\Local\Temp\jieba.cache
Loading model cost 0.434 seconds.
Prefix dict has been built successfully.


In [6]:

# 计算评论情感值

# 分句处理，用于统计每一句的情感值，评论的情感值等于各分句情感值的和
def cut_sentence(sentence):
    sentences = SentenceSplitter.split(sentence)
    sentence_list = [ w for w in sentences]
    return sentence_list

# 读取正面情感词列表和正面评价词列表
pos_sense_file = open("../static/Sensiword_cnki/正面情感词语（中文）.txt",'r+', encoding="utf-8")
pos_sense_list = [word.replace(" \n", "") for word in pos_sense_file.readlines()]
pos_comment_file = open("../static/Sensiword_cnki/正面评价词语（中文）.txt",'r+', encoding="utf-8")
pos_comment_list = [word.replace(" \n", "") for word in pos_comment_file.readlines()]

# 定义match_pos函数，用于匹配正面情感词和正面评价词
def match_pos(word):
    pos_list = pos_sense_list + pos_comment_list
    if word in pos_list:
        return True
    else:
        return False

# 读取负面情感词列表和负面评价词列表
neg_sense_file = open("../static/Sensiword_cnki/负面情感词语（中文）.txt",'r+', encoding="utf-8")
neg_sense_list = [word.replace(" \n", "") for word in neg_sense_file.readlines()]
neg_comment_file = open("../static/Sensiword_cnki/负面评价词语（中文）.txt",'r+', encoding="utf-8")
neg_comment_list = [word.replace(" \n", "") for word in neg_comment_file.readlines()]

# 定义match_neg函数，用于匹配负面情感词和负面评价词
def match_neg(word):
    neg_list = neg_sense_list + neg_comment_list
    if word in neg_list:
        return True
    else:
        return False

# 读取程度级别词列表
most_adv_list = [word.replace("\n", "") for word in open("../static/Sensiword_cnki/程度级别词语（中文）_most.txt", 'r+', encoding="utf-8").readlines()]
very_adv_list = [word.replace("\n", "") for word in open("../static/Sensiword_cnki/程度级别词语（中文）_very.txt", 'r+', encoding="utf-8").readlines()]
more_adv_list = [word.replace("\n", "") for word in open("../static/Sensiword_cnki/程度级别词语（中文）_more.txt", 'r+', encoding="utf-8").readlines()]
ish_adv_list = [word.replace("\n", "") for word in open("../static/Sensiword_cnki/程度级别词语（中文）_ish.txt", 'r+', encoding="utf-8").readlines()]
insufficiently_adv_list = [word.replace("\n", "") for word in open("../static/Sensiword_cnki/程度级别词语（中文）_insufficiently.txt", 'r+', encoding="utf-8").readlines()]
over_adv_list = [word.replace("\n", "") for word in open("../static/Sensiword_cnki/程度级别词语（中文）_over.txt", 'r+', encoding="utf-8").readlines()]

# 定义match_degree函数，用于匹配程度级别词并返回相应的权值
def match_adv(word):
    if word in most_adv_list:      #最高级权重
        return 8
    elif word in very_adv_list:    #比较级权重
        return 6
    elif word in more_adv_list:    #比较级权重
        return 4
    elif word in ish_adv_list:     #轻微程度词权重
        return 2
    elif word in insufficiently_adv_list:   #相对程度词权重
        return 0.5
    elif word in over_adv_list:    #过度权重
        return -2
    else:                      # 无程度词
        return 1

# 定义calculate_sentiment函数，用于计算单个句子的情感值
not_list = [word.replace("\n", "") for word in open("../static/Sensiword_cnki/否定词.txt", 'r+', encoding="utf-8").readlines()]

def match_not(word):
    if word in not_list:
        return True
    else:
        return False

# 计算每条评论的情感得分
all_adv_list = [word.replace("\n", "") for word in open("../static/Sensiword_cnki/程度级别词语（中文）.txt", 'r+', encoding="utf-8").readlines()]
def calculate_sentiment_score(sentence):
    sub_sentence_score_sum= 0    # 评论每个分句的情感得分
    sub_sentences = cut_sentence(sentence)   # 对评论进行分句
    for sub_sentence in sub_sentences:       # 逐个计算分句的情感得分
        # 对分句进行分词
        word_list = list(jieba.cut(sub_sentence, cut_all=False)) # 精确模式
        # 去除停用词但是不包括程度词
        cleaned_word_list = []
        for word in word_list:
            if word in stopwordsList:
                if word in all_adv_list:
                    cleaned_word_list.append(word)
            else:
                cleaned_word_list.append(word)
        i = 0       #记录扫描到的词位子
        s = 0       #记录情感词的位置
        posscore = 0 #记录积极情感得分
        negscore = 0 #记录消极情感得分
        for word in cleaned_word_list:
            if match_pos(word):  # 找到积极情感词
                # 在情感词前面寻找程度副词
                weight = 1
                for w in cleaned_word_list[s:i]:
                    if match_adv(w) != 1:
                        weight = match_adv(w)
                        break
                # 根据是否存在否定词来判断情感得分
                if match_not(cleaned_word_list[i-1]):   # 若积极情感词前存在否定词，则消极情绪加分
                    negscore = negscore + 1 * weight
                else:                                   # 否则积极情绪加分
                    posscore = posscore + 1 * weight
                s = i + 1  # 记录情感词位置
            elif match_neg(word):  # 找到消极情感词
                weight = 1
                for w in cleaned_word_list[s:i]:
                    if match_adv(w) != 1:
                        weight = match_adv(w)
                        break
                # 根据是否存在否定词来判断情感得分
                if match_not(cleaned_word_list[i-1]):   # 若消极情感词前存在否定词，则积极情绪加分
                    posscore = posscore + 1 * weight
                else:                                   # 否则消极情绪加分
                    negscore = negscore + 1 * weight
                s = i + 1  # 记录情感词位置
            elif word =='!' or  word =='！' or word =='?' or word == '？':  # 符号加强情感强度
                if posscore > negscore:
                    posscore += 2           # 若此分句整体的积极情绪值大于消极情绪值，则符号加强积极情绪值
                elif negscore > posscore:
                    negscore += 2
            i += 1
        sub_sentence_score = posscore - negscore
        sub_sentence_score_sum += sub_sentence_score
    return sub_sentence_score_sum

#将每条评论的情感得分添加到新列中。
OriginalData_DF["评论情感得分"] = OriginalData_DF["干净文本"].apply(calculate_sentiment_score)



In [7]:
# 保存文件
tempDF = OriginalData_DF
tempDF.drop(labels=['用户发表的关于文心一言或ChatGPT的评论性文本',"干净文本"], axis=1, inplace=True)
OriginalData_DF.to_csv("../merged_file_2.csv", encoding="utf-8", index=False)
